# EDGAR Financial Sentiment — Part 7: Structured Extraction (JSON + Chain-of-Thought)

The last Track B part. So far every method produced **one label** per text. Here the LLM returns a **structured record** — sentiment *and* guidance direction *and* key figures *and* a rationale — as JSON, reasoning step by step first. This is the on-ramp to real **8-K earnings releases**.

**What you'll practice (the core concept):** **structured-output prompting** (`build_extraction_prompt`) and **robust JSON parsing** (`parse_json`) — those are your blanks. You'll then *measure* whether chain-of-thought + a schema actually help, against a minimal prompt.

> **Run in Google Colab with a T4 GPU.** Uses the gated Mistral model (Parts 4–6 login). Prompts are original and finance-specific (kept separate from the ungraded Assignment 4).

## 0. Why structured extraction? From a label to a record

A single sentiment label throws away most of what a financial statement says. An earnings release tells you **which way guidance moved**, **what EPS and revenue were**, and **why** — a whole record, not a mood. **Structured extraction** asks the model to return that record as **JSON** instead of a word:

```json
{"sentiment": "positive", "guidance_direction": "raised",
 "key_figures": ["EPS $1.42", "revenue $2.8B"], "rationale": "..."}
```

That's something you can put in a database, regress on, or chart — and it's the on-ramp to **real 8-K EX-99.1 earnings releases** (the data pipeline in `PROJECT_SCOPE.md`).

**Two new skills + one technique:**
- **Structured-output prompting** — getting the model to emit a *parseable* object that follows a schema.
- **Robust JSON parsing** — LLMs wrap JSON in prose, markdown fences, or trailing commas; your parser must survive that and fail gracefully (return `None`) when there's no JSON.
- **Chain-of-thought (CoT)** — "reason first, then answer" usually improves the answer. We'll *measure* whether it does.

**What we measure:** the **validity rate** (did we get parseable JSON?) and agreement with a small hand-labeled gold set — your CoT+schema prompt vs. a minimal one. You'll also see that **objective** fields (guidance direction, figures) extract far more reliably than the **subjective** overall sentiment — a real, useful finding.

## 1. Setup

In [ ]:
!pip install -q -U transformers bitsandbytes accelerate

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import json, re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
assert device.type == 'cuda', 'Switch the Colab runtime to a T4 GPU before running.'

### Hugging Face login (Mistral is gated)
Same as Parts 4–6: accept the license, make a token, log in.

In [ ]:
from huggingface_hub import login
login()   # paste your HF token when prompted

## 2. Config & the target schema

In [ ]:
torch.manual_seed(10)

MODEL_ID   = 'mistralai/Mistral-7B-Instruct-v0.3'

# The schema we want the model to fill, and the allowed values for the two we can score:
SENTIMENTS = ['positive', 'neutral', 'negative']
GUIDANCE   = ['raised', 'lowered', 'maintained', 'none']

## 3. The data — press-release excerpts + a small gold set  *(provided)*

Six short, realistic earnings-statement excerpts, each hand-labeled with the *overall sentiment* and the *guidance direction* so we can score the extraction. (These are press-release-**style** text; the real target is 8-K EX-99.1 filings — see Section 9.)

In [ ]:
EXCERPTS = [
    {'text': 'Helios Semiconductor today reported third-quarter diluted EPS of $1.42, ahead of its prior '
             'outlook, on revenue of $2.8 billion, up 18% year over year. Citing strong demand for its '
             'data-center products, the company raised its full-year revenue guidance to a range of $11.2 '
             'to $11.5 billion.',
     'gold': {'sentiment': 'positive', 'guidance_direction': 'raised'}},
    {'text': 'Meridian Retail Group reported a fourth-quarter net loss of $0.31 per share as comparable-store '
             'sales fell 6%. Management pointed to soft consumer spending and elevated markdowns, and lowered '
             'its full-year earnings guidance to $0.90-$1.05 per share from the prior $1.30-$1.45.',
     'gold': {'sentiment': 'negative', 'guidance_direction': 'lowered'}},
    {'text': 'Cascade Utilities announced second-quarter earnings of $0.88 per share, in line with analyst '
             'expectations, on revenue of $1.1 billion. The company reaffirmed its full-year guidance of '
             '$3.40 to $3.55 per share.',
     'gold': {'sentiment': 'neutral', 'guidance_direction': 'maintained'}},
    {'text': 'Orion Biotech delivered first-quarter revenue of $410 million, beating consensus, and EPS of '
             '$0.22 versus a loss a year ago. However, management cautioned that pricing pressure would weigh '
             'on the back half and trimmed its full-year operating margin outlook.',
     'gold': {'sentiment': 'negative', 'guidance_direction': 'lowered'}},   # mixed: a beat, but a cut outlook
    {'text': 'Granite Industrial said it completed the acquisition of Pinnacle Tooling for $640 million in '
             'cash, expanding its presence in precision components. The transaction closed in the fourth '
             'quarter.',
     'gold': {'sentiment': 'neutral', 'guidance_direction': 'none'}},
    {'text': 'Aurora Software reported record quarterly bookings and EPS of $0.67, up from $0.51, driven by '
             '30% growth in its cloud segment. The company did not update its annual forecast.',
     'gold': {'sentiment': 'positive', 'guidance_direction': 'none'}},
]
print(len(EXCERPTS), 'excerpts loaded')

## 4. Load Mistral-7B (4-bit) + a generation helper  *(provided)*

Causal LM, same as Parts 5–6. Note `generate` is **greedy** (`do_sample=False`) here — for extraction we want deterministic, reproducible output, not the diversity we wanted when *generating* data in Part 6.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map={'': 0})
model.eval()
print('Loaded', MODEL_ID)

In [ ]:
@torch.no_grad()
def generate(prompt_text, max_new_tokens=256):
    messages = [{'role': 'user', 'content': prompt_text}]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors='pt').to(device)
    out = model.generate(inputs, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True).strip()

## 5. Write the extraction prompt  &larr; **your code**

`build_extraction_prompt(text)` should ask the model to **reason briefly, then emit one JSON object** with these keys:
- `"sentiment"` — one of `positive` / `neutral` / `negative`
- `"guidance_direction"` — one of `raised` / `lowered` / `maintained` / `none`
- `"key_figures"` — a list of short strings like `"EPS $1.42"`
- `"rationale"` — a one-sentence justification

Good structured prompts: **name every key and its allowed values**, **show the JSON shape**, ask for **valid JSON only** (double quotes, no trailing commas), and put the chain-of-thought *before* the JSON. The clearer the format demand, the easier your `parse_json` has it.

In [ ]:
def build_extraction_prompt(text):
    """Reason step by step, then emit one JSON object following the schema."""
    ### YOUR CODE HERE
    instructions = (
        'You are a financial analyst. Read the company statement and extract structured information.\n'
        'First, think step by step in one or two sentences. THEN output a SINGLE JSON object with exactly '
        'these keys:\n'
        '  "sentiment": one of "positive", "neutral", "negative" (overall tone toward the company)\n'
        '  "guidance_direction": one of "raised", "lowered", "maintained", "none"\n'
        '  "key_figures": a list of short strings for reported numbers (e.g. "EPS $1.42", "revenue $2.8B")\n'
        '  "rationale": a one-sentence justification\n'
        'Output valid JSON only for that object: double quotes, no trailing commas.'
    )
    return instructions + '\n\nStatement:\n' + text
    ### END YOUR CODE

### See your prompt + a raw reply
Run after filling in `build_extraction_prompt`. Notice the model reasons, *then* writes JSON — your parser has to find the JSON inside that.

In [ ]:
print('=== PROMPT ===')
print(build_extraction_prompt(EXCERPTS[0]['text']))
print('\n=== MODEL REPLY ===')
print(generate(build_extraction_prompt(EXCERPTS[0]['text'])))

## 6. Parse the JSON robustly  &larr; **your code**

`parse_json(reply)` returns a **dict**, or **`None`** if there's no usable JSON. The model's reply has reasoning prose (and sometimes ```` ```json ```` fences) around the object, so you can't just `json.loads` the whole thing. A robust trick: take everything from the **first `{`** to the **last `}`** and try to parse that; return `None` if it fails. Returning `None` (instead of crashing) is what lets you *count* failures as the validity rate.

In [ ]:
def parse_json(reply):
    """Extract the JSON object from the reply; return a dict, or None if unparseable."""
    ### YOUR CODE HERE
    start, end = reply.find('{'), reply.rfind('}')
    if start == -1 or end == -1 or end < start:
        return None
    try:
        return json.loads(reply[start:end + 1])
    except json.JSONDecodeError:
        return None
    ### END YOUR CODE

### Test your parser
It should pull the dict out of the prose, and return `None` when there's no JSON.

In [ ]:
_messy = 'Here is the result: {"sentiment": "positive", "guidance_direction": "raised", '\
         '"key_figures": ["EPS $1.42"]} -- hope that helps!'
print(parse_json(_messy))                                  # -> dict
print(parse_json('the model only wrote prose, no JSON'))   # -> None

## 7. Extract from every excerpt  *(provided driver)*
Run your prompt + parser over all six and read the structured output. This is the qualitative look before we score.

In [ ]:
for ex in EXCERPTS:
    obj = parse_json(generate(build_extraction_prompt(ex['text'])))
    print('TEXT:', ex['text'][:85], '...')
    print('EXTRACTED:', json.dumps(obj, indent=2) if obj else '<<invalid / no JSON>>')
    print('GOLD:', ex['gold'], '\n' + '-' * 70)

#### ✅ Reflect — read the records
Look at what came back. Two patterns usually show up: the **objective** fields are nearly always right — `guidance_direction` and `key_figures` are stated explicitly in the text, so the model just has to copy them. The **subjective** `sentiment` is shakier, especially on the mixed excerpt (Orion: a *beat* but a *cut outlook*) where even humans disagree. And occasionally a reply isn't valid JSON at all — that's the failure mode a downstream pipeline cares about most, because an unparseable record is simply unusable.

## 8. Measure — does CoT + a schema actually help?

#### ✅ A reasonable prediction
Your structured+CoT prompt should produce **valid JSON nearly every time** (the schema and 'JSON only' demand pin the format), and should get **guidance_direction** almost perfectly while **sentiment** trails (it's subjective). The **minimal** prompt — no schema, no format demand — will likely have a **lower validity rate** (more prose, fewer parseable objects), which drags everything down: you can't score a record you can't parse. That validity gap is the main point.

In [ ]:
def build_extraction_prompt_minimal(text):
    """A terse contrast prompt: no step-by-step, no schema, no format demand."""
    return 'Give the sentiment and guidance direction of this as JSON:\n' + text

def evaluate_extraction(prompt_fn, label):
    valid = sent_ok = guid_ok = 0
    n = len(EXCERPTS)
    for ex in EXCERPTS:
        obj = parse_json(generate(prompt_fn(ex['text'])))
        if obj is None:
            continue
        valid += 1
        if str(obj.get('sentiment', '')).lower() == ex['gold']['sentiment']:
            sent_ok += 1
        if str(obj.get('guidance_direction', '')).lower() == ex['gold']['guidance_direction']:
            guid_ok += 1
    print(f'  {label:22s} valid {valid}/{n} | sentiment {sent_ok}/{n} | guidance {guid_ok}/{n}')
    return valid, sent_ok, guid_ok

print('Your structured + CoT prompt vs a minimal prompt:')
evaluate_extraction(build_extraction_prompt, 'yours (CoT + schema)')
evaluate_extraction(build_extraction_prompt_minimal, 'minimal')

#### ✅ Reflect — what the numbers say
- **Validity is the gatekeeper.** The CoT+schema prompt should parse cleanly far more often; the minimal one loses records to prose or malformed JSON, and an unparseable record scores zero on *every* field. Most of the 'accuracy' difference between prompts is really a *format* difference — exactly the lesson from Part 5's wording experiment, now with structure.
- **Objective beats subjective.** `guidance_direction` and `key_figures` are *stated* in the text → high agreement. `sentiment` is a *judgment* → lower, and most of its errors cluster on genuinely mixed releases. This is why serious systems extract the hard facts (the surprise, the guidance) structurally and treat overall 'sentiment' as the soft, noisy signal it is.
- **CoT helps modestly here** because the task is short; it helps more as the reasoning gets longer (multi-step figures, conditional guidance).

## 9. On-ramp to real 8-K data

The excerpts here are press-release-*style* text. The real target is **8-K EX-99.1 earnings releases** pulled from EDGAR — the heavier data pipeline described in `PROJECT_SCOPE.md` (SEC `submissions` API + `Archives`, respecting fair-access limits and a declared User-Agent). **Your extraction code doesn't change** — you'd feed it real filing text instead of these examples. That swap (clean examples → long, messy real filings) is exactly where structured extraction earns its keep: the validity rate drops, the figures get harder to pull, and the measurement harness you just wrote becomes the thing that tells you how well it's working at scale.

## 10. Recap & next

You turned the model into a structured extractor: a CoT + schema prompt, a robust JSON parser that fails gracefully, and a measured comparison that showed **validity is the gatekeeper** and **objective fields extract more reliably than subjective sentiment**. That record — not a bare label — is what real financial NLP systems actually consume.

**That completes Track B (Parts 5–7).** Next is the convergence:

**Part 8 — the bake-off:** one results table scoring *every* approach you've built — lexicon, fine-tuned GPT-2, BERT vs FinBERT, QLoRA Mistral, zero/few-shot LLM, synthetic-augmented — on a single held-out set. The portfolio centerpiece, and where the `Choosing_a_Training_Method` scoreboard finally fills in.